In [ ]:
from pyspark.sql import SparkSession
from pathlib import Path

# Initialize a Spark session
spark = SparkSession.builder.appName("FragranceDataCleaning").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Define the file path
relative_path = "data/frag_raw.csv"
current_dir = Path().resolve()
file_path = current_dir / relative_path

try:
    frag_raw_df = spark.read.csv(str(file_path), header=True, inferSchema=True)
    frag_raw_df.createOrReplaceTempView("frag_raw")

    perfume_cleaned_df = spark.sql("""
    WITH ranked AS (
        SELECT
            REGEXP_EXTRACT(url, '([a-zA-Z0-9]+)\\.html$', 1) as id,
            replace(regexp_extract(url, '/perfume/[^/]+/([^/]+)-[0-9]+\\.html', 1), '-', ' ') as name,
            replace(regexp_extract(url, '/perfume/([^/]+)/', 1), '-', ' ') as brand,
            REGEXP_EXTRACT(description, 'was launched in ([0-9]{4})', 1) as release_year,
            replace(replace(replace(perfumers, '[', ''), ']', ''),"'", "") as perfumers,
            CASE
                WHEN name LIKE '%for women and men' THEN 'unisex'
                WHEN name LIKE '%for women' THEN 'women'
                WHEN name LIKE '%for men' THEN 'men'
                ELSE NULL
            END as gender,
            TRY_CAST(REGEXP_REPLACE(rating_count, ',', '') AS INT) as rating_count,
            TRY_CAST(rating AS DECIMAL(10,2)) as rating,
            replace(replace(replace(main_accords, '[', ''), ']', ''),"'", "") as accords,
            lower(replace(regexp_extract(description, '(?i)top note[s]? (is|are) ([^.;]*)', 2), ' and', ',')) as top_notes,
            lower(replace(regexp_extract(description, '(?i)middle note[s]? (is|are) ([^.;]*)', 2), ' and', ',')) as mid_notes,
            lower(replace(regexp_extract(description, '(?i)base note[s]? (is|are) ([^.;]*)', 2), ' and', ',')) as base_notes,
            description,
            url,
            row_number() over (partition by url order by rating_count desc) as rn
        FROM frag_raw
    )
    SELECT 
        *,
        TRIM(
        CONCAT(
            CASE WHEN accords IS NOT NULL AND accords != ''
                THEN ' ' || array_join(
                    transform(split(accords, ','), x -> concat('accords_', replace(trim(x), ' ', '_'))),
                    ' '
                )
                ELSE ''
            END,
            CASE WHEN top_notes IS NOT NULL AND top_notes != ''
                THEN ' ' || array_join(
                    transform(split(top_notes, ','), x -> concat('top_notes_', replace(trim(x), ' ', '_'))),
                    ' '
                )
                ELSE ''
            END,
            CASE WHEN mid_notes IS NOT NULL AND mid_notes != ''
                THEN ' ' || array_join(
                    transform(split(mid_notes, ','), x -> concat('mid_notes_', replace(trim(x), ' ', '_'))),
                    ' '
                )
                ELSE ''
            END,
            CASE WHEN base_notes IS NOT NULL AND base_notes != ''
                THEN ' ' || array_join(
                    transform(split(base_notes, ','), x -> concat('base_notes_', replace(trim(x), ' ', '_'))),
                    ' '
                )
                ELSE ''
            END
        )
    ) AS perfume_string
    FROM ranked
    WHERE rn = 1
    """)

    perfume_cleaned_df.show(truncate=False)

    # print("Schema:")
    # perfume_cleaned_df.printSchema()

    # print("\n ======================================= \n Data Types:")
    # print(perfume_cleaned_df.dtypes)

    # # Save to Delta table
    # perfume_cleaned_df.write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable("fragrance_db.default.fragrance_cleaned")
    # print("Delta table updated!")

    perfume_cleaned_df.toPandas().to_csv("frag_cleaned_with_texts.csv", index=False)
    print("\n======================================= \n\nCleaned data written to frag_cleaned_with_texts.csv")

except Exception as e:
    print(f"An error occurred: {e}")


+-----+-----------------------+------------------------+------------+--------------------+------+------------+------+-------------------------------------------------------------------------------------------+------------------------------------------------------------+---------------------------------------------------------+----------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------+---+----------------------------------------------------------------------------------------------

In [ ]:
# Convert accords and notes into json object

from pyspark.sql.functions import split, trim, to_json, col, regexp_replace

def to_json_array(df, col_name, new_col_name):
    return df.withColumn(
        new_col_name,
        to_json(
            split(
                trim(regexp_replace(col(col_name), r"[\[\]']", "")),
                r",\s*"
            )
        )
    )

# Apply to all relevant columns
perfume_cleaned_df_json = to_json_array(perfume_cleaned_df, "main_accords", "main_accords_json")
perfume_cleaned_df_json = to_json_array(perfume_cleaned_df_json, "top_notes", "top_notes_json")
perfume_cleaned_df_json = to_json_array(perfume_cleaned_df_json, "mid_notes", "mid_notes_json")
perfume_cleaned_df_json = to_json_array(perfume_cleaned_df_json, "base_notes", "base_notes_json")

perfume_cleaned_df_json.show(n=10, truncate=False)


perfume_cleaned_df_json.toPandas().to_csv("frag_cleaned_json_notes.csv", index=False)
print("\n======================================= \n\nCleaned data written to frag_cleaned_json_notes.csv")

print("\n======================================= \n\nData Types:")
print(perfume_cleaned_df_json.dtypes)


+-----+-----------------------+------------------------+------------+----------------+------+------------+------+-------------------------------------------------------------------------------------------+------------------------------------------------------------+---------------------------------------------------------+----------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------+---+--------------------------------------------------------------------------------------------------

In [ ]:
# Get all unique accords and notes from the JSON columns
from pyspark.sql.functions import explode, from_json, array_distinct
from pyspark.sql.types import ArrayType, StringType

# Parse JSON arrays back to Spark arrays and explode
def get_unique_from_json_col(df, json_col):
    return (
        df
        .withColumn("arr", from_json(col(json_col), ArrayType(StringType())))
        .select(explode(col("arr")).alias("item"))
        .distinct()
        .select("item")
    )

# Collect unique values from each column
main_accords_unique = get_unique_from_json_col(perfume_cleaned_df_json, "main_accords_json")
top_notes_unique = get_unique_from_json_col(perfume_cleaned_df_json, "top_notes_json")
mid_notes_unique = get_unique_from_json_col(perfume_cleaned_df_json, "mid_notes_json")
base_notes_unique = get_unique_from_json_col(perfume_cleaned_df_json, "base_notes_json")

# Union all and get unique values
all_unique = (
    main_accords_unique
    .union(top_notes_unique)
    .union(mid_notes_unique)
    .union(base_notes_unique)
    .distinct()
    .orderBy("item")
)

all_unique_list = [row.item for row in all_unique.collect()]
print("\n======================================= \n \nAll Unique Accords and Notes:")
print(all_unique_list)
print(f"Number of distinct items: {len(all_unique_list)}")


 
All Unique Accords and Notes:
['', 'Champagne', 'Pear', 'absinthe', 'acai berry', 'accord eudora®', 'acerola', 'acerola blossom', 'acetylfuran', 'acácia', 'african freesia petals', 'african geranium', 'african ginger', 'african orange flower', 'african violet', 'agarwood', 'agarwood (oud)', 'agave', 'agave nectar', 'aglaia', 'akigalawood', 'albizia', 'alcohol', 'aldehydes', 'aldehydic', 'aldron', 'algae', 'algerian geranium', 'allspice', 'almond', 'almond blossom', 'almond cream', 'almond milk', 'almond tree', 'almond wood', 'aloe vera', 'alpinia', 'althaea', 'aluminum', 'alumroot', 'alyssum', 'amalfi lemon', 'amaranth', 'amaretto', 'amaryllis', 'amazon lily', 'amber', 'amber from tunis', 'amber oil', 'amber xtreme', 'ambergris', 'ambertonic', 'amberwood', 'ambrarome', 'ambreine', 'ambretone', 'ambrette', 'ambrette (musk mallow)', 'ambrettolide', 'ambrinol', 'ambrocenide', 'ambrofix™', 'ambrostar™', 'ambrox super', 'ambroxan', 'american apple', 'ammophila (beach grass)', 'amyl salic

In [41]:
import pandas as pd
import json

# Write the unique accords and notes to a JSON file
with open("all_unique_accords_and_notes.json", "w", encoding="utf-8") as f:
    json.dump(all_unique_list, f, ensure_ascii=False, indent=2)

print("Unique accords and notes written to all_unique_accords_and_notes.json")


Unique accords and notes written to all_unique_accords_and_notes.json


In [ ]:
from pyspark.sql import SparkSession

# Initialize a Spark session
spark = SparkSession.builder.appName("FragranceDataCleaning").getOrCreate()
from datetime import datetime

# Get current timestamp when stopping Spark
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
spark.stop()
print(f"Spark session stopped at {timestamp}")

Spark session stopped at 2025-09-21 20:19:16
